In [1]:
# =======================
# Imports
# =======================
import os
from pathlib import Path

import numpy as np
import pandas as pd

from bertopic import BERTopic
from scipy.spatial.distance import pdist

# Optional (only if you want to auto-detect best model name from grid results)
# import pandas as pd

# =======================
# Parameters you can tweak
# =======================
# Level depth as used across your project
level_depth = "0"

# Option A (recommended): load the FINAL best model saved by your "selection" script
#   It expects ../results/levels/level_<lvl>/best/best_model (a directory)
USE_FINAL_BEST = True

# If USE_FINAL_BEST=False, Option B will be used:
#   - set the exact embedding model name you used to generate the embeddings
#   - set the UMAP/HDBSCAN params to rebuild the model directory suffix
embedding_model_name = "all-MiniLM-L6-v2"  # e.g. "all-MiniLM-L6-v2" or "paraphrase-MiniLM-L6-v2"
umap_cfg = {'n_components': 5, 'n_neighbors': 25, 'min_dist': 0.0}
hdbscan_cfg = {'min_cluster_size': 50}

# =======================
# Project paths
# =======================
BASE_DIR = Path(f"../../results/levels/level_{level_depth}")
GRID_DIR = BASE_DIR / "grid_search"

DF_SAMPLED_CSV   = GRID_DIR / f"df_sampled_level_{level_depth}.csv"
EMBED_DIR        = GRID_DIR / f"embeddings_level_{level_depth}"
BEST_DIR         = BASE_DIR / "best"            # contains "best_model" dir after your selection script
BEST_MODEL_DIR   = BEST_DIR / "best_model"      # directory saved by BERTopic.save()

# Embeddings path (for diagnostics we need the SAME embedding model used to fit the topic model)
EMBEDDINGS_NPY   = EMBED_DIR / f"embedding_{embedding_model_name}_level_{level_depth}.npy"

# Option B: reconstruct model folder name if not using BEST/
MODEL_SUFFIX = (
    f"{embedding_model_name}_"
    f"umap{umap_cfg['n_components']}_"
    f"umap{umap_cfg['n_neighbors']}_"
    f"umap{umap_cfg['min_dist']}_"
    f"hdbscan{hdbscan_cfg['min_cluster_size']}"
)
MODEL_DIR_OPT_B = GRID_DIR / f"bertopic_models_level_{level_depth}" / MODEL_SUFFIX

# =======================
# Load df_sampled, topic_model, embeddings
# =======================
if not DF_SAMPLED_CSV.exists():
    raise FileNotFoundError(f"Missing df_sampled at: {DF_SAMPLED_CSV}")

df_sampled = pd.read_csv(DF_SAMPLED_CSV)

# Load topic model (prefer final best if available)
if USE_FINAL_BEST and BEST_MODEL_DIR.exists():
    print(f"[INFO] Loading topic model from BEST: {BEST_MODEL_DIR}")
    topic_model = BERTopic.load(str(BEST_MODEL_DIR))
else:
    if not MODEL_DIR_OPT_B.exists():
        raise FileNotFoundError(
            f"Model directory not found.\n"
            f"Tried BEST: {BEST_MODEL_DIR.exists()} at {BEST_MODEL_DIR}\n"
            f"Tried Option B: {MODEL_DIR_OPT_B}"
        )
    print(f"[INFO] Loading topic model from: {MODEL_DIR_OPT_B}")
    topic_model = BERTopic.load(str(MODEL_DIR_OPT_B))

# Load embeddings (matching the embedding model used to fit)
if not EMBEDDINGS_NPY.exists():
    raise FileNotFoundError(f"Embeddings not found at: {EMBEDDINGS_NPY}\n"
                            f"Make sure 'embedding_model_name' matches the embeddings you generated.")
embeddings = np.load(EMBEDDINGS_NPY)

print("[INFO] Loaded:")
print(" - df_sampled:", DF_SAMPLED_CSV)
print(" - embeddings:", EMBEDDINGS_NPY, embeddings.shape)
print(" - model loaded OK")

# =======================
# DIAGNOSTICS (your block)
# =======================
print("\n=== DIAGNOSTICS ===")
print("NUM DOCS:", len(df_sampled))

# 1) alignment
print("Emb shape:", embeddings.shape)
if len(embeddings) != len(df_sampled):
    print("WARNING: embeddings length != docs length")

# 2) labels distribution
labels = np.array(topic_model.topics_)
unique, counts = np.unique(labels, return_counts=True)
print("labels counts sample:", dict(zip(unique, counts)))
n_topics = len([u for u in unique if u != -1])
print("n_topics (excl -1):", n_topics)
print("outliers %:", (labels == -1).mean() * 100)

# 3) embedding variance + pairwise distances (sample)
print("emb mean, std:", embeddings.mean(), embeddings.std())
zerodims = (embeddings.std(axis=0) < 1e-6).sum()
print("num zero-std dims:", zerodims, "of", embeddings.shape[1])

# To keep runtime reasonable, sample up to 1000 items for pairwise distances
sample_size = min(1000, len(embeddings))
if sample_size >= 2:
    idx = np.random.choice(len(embeddings), size=sample_size, replace=False)
    d = pdist(embeddings[idx], metric='cosine')
    print("pairwise cosine dist - min, median, max:", d.min(), np.median(d), d.max())
else:
    print("pairwise cosine dist: skipped (not enough samples)")

# 4) text stats
lens = df_sampled['text_preprocessed'].str.split().str.len().fillna(0)
print("median tokens:", lens.median(), "pct short <3:", (lens < 3).mean() * 100)
print("duplicates:", len(df_sampled) - df_sampled['text_preprocessed'].nunique())

# 5) suggestions
if (labels == -1).mean() > 0.3:
    print("SUGGESTION: try reducing hdbscan min_cluster_size (e.g. 10, 20, 50) and/or min_samples")
# Median computed only if we computed 'd' above
if sample_size >= 2 and np.median(d) < 0.05:
    print("SUGGESTION: embeddings look too similar (median pairwise cosine < 0.05). "
          "Check embedding generation or try a different SentenceTransformer model.")


/home/students/s328743/.conda/envs/bertopic_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[INFO] Loading topic model from: ../../results/levels/level_0/grid_search/bertopic_models_level_0/all-MiniLM-L6-v2_umap5_umap25_umap0.0_hdbscan50
[INFO] Loaded:
 - df_sampled: ../../results/levels/level_0/grid_search/df_sampled_level_0.csv
 - embeddings: ../../results/levels/level_0/grid_search/embeddings_level_0/embedding_all-MiniLM-L6-v2_level_0.npy (30054, 384)
 - model loaded OK

=== DIAGNOSTICS ===
NUM DOCS: 30054
Emb shape: (30054, 384)
labels counts sample: {-1: 14576, 0: 1090, 1: 721, 2: 658, 3: 611, 4: 557, 5: 520, 6: 510, 7: 484, 8: 470, 9: 453, 10: 376, 11: 369, 12: 367, 13: 365, 14: 320, 15: 285, 16: 263, 17: 250, 18: 250, 19: 246, 20: 245, 21: 242, 22: 238, 23: 226, 24: 221, 25: 208, 26: 203, 27: 193, 28: 191, 29: 180, 30: 176, 31: 164, 32: 163, 33: 162, 34: 161, 35: 159, 36: 154, 37: 145, 38: 139, 39: 137, 40: 136, 41: 132, 42: 124, 43: 123, 44: 121, 45: 117, 46: 104, 47: 103, 48: 101, 49: 98, 50: 98, 51: 95, 52: 87, 53: 85, 54: 81, 55: 73, 56: 71, 57: 70, 58: 69, 59: 68,